In [1]:
import os
import torch
import torch.nn as nn
from torch.ao.quantization import FakeQuantize, MinMaxObserver
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)
torch.set_num_threads(2)
torch.backends.quantized.engine = "fbgemm"

fake_quant = FakeQuantize(observer=MinMaxObserver, quant_min=0, quant_max=255)
fake_quant.enable_observer()
fake_quant.enable_fake_quant()

calib_x = torch.tensor([[-2.0, -1.0, 0.0, 1.0, 2.0]])
fake_quant(calib_x)

x_new = torch.tensor([[0.4, 1.2, -0.9, 0.0]])
x_q = fake_quant(x_new)
print("Input:", x_new)
print("Fake quantized output:", x_q)
print("Max rounding error:", (x_new - x_q).abs().max().item())

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),
])

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
print(f"Train samples: {len(train_ds)} | Test samples: {len(test_ds)}")

Input: tensor([[ 0.4000,  1.2000, -0.9000,  0.0000]])
Fake quantized output: tensor([[ 0.3922,  1.1922, -0.8941,  0.0000]])
Max rounding error: 0.00784313678741455


100%|██████████| 9.91M/9.91M [00:02<00:00, 3.91MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 123kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.40MB/s]

Train samples: 60000 | Test samples: 10000


In [2]:
class SimpleQATModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.fc2(self.relu(self.fc1(x)))
        return self.dequant(x)

model = SimpleQATModel()
model.train()

model_fused = torch.ao.quantization.fuse_modules(model, [["fc1", "relu"]])
model_fused.qconfig = torch.ao.quantization.get_default_qat_qconfig("fbgemm")
torch.ao.quantization.prepare_qat(model_fused, inplace=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fused.parameters(), lr=1e-3)

epochs = 3
for epoch in range(epochs):
    model_fused.train()
    if epoch == 2:
        model_fused.apply(torch.ao.quantization.disable_observer)
    total_loss = 0.0
    for data, target in train_loader:
        optimizer.zero_grad(set_to_none=True)
        output = model_fused(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss / len(train_loader):.4f}")

torch.save(model_fused.state_dict(), "qat_prepared.pt")
print("Saved QAT-trained checkpoint to qat_prepared.pt")

/tmp/ipykernel_791/4018730000.py:20: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare_qat(model_fused, inplace=True)
/opt/conda/lib/python3.12/site-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future re

Epoch 1/3 | Loss: 0.3175
Epoch 2/3 | Loss: 0.1168
Epoch 3/3 | Loss: 0.0773
Saved QAT-trained checkpoint to qat_prepared.pt


In [3]:
import time

def evaluate(m, loader):
    m.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            preds = m(data).argmax(dim=1)
            correct += (preds == target).sum().item()
    return correct / len(loader.dataset)

def model_size_kb(m):
    torch.save(m.state_dict(), "_tmp.pt")
    size = os.path.getsize("_tmp.pt") / 1024
    os.remove("_tmp.pt")
    return size

def measure_latency_ms(m, runs=50):
    m.eval()
    sample = next(iter(test_loader))[0]
    with torch.no_grad():
        for _ in range(5):
            m(sample)
        t0 = time.perf_counter()
        for _ in range(runs):
            m(sample)
    return (time.perf_counter() - t0) / runs * 1000

fp32_model = SimpleQATModel()
fp32_model.train()
fp32_opt = torch.optim.Adam(fp32_model.parameters(), lr=1e-3)
for epoch in range(3):
    for data, target in train_loader:
        fp32_opt.zero_grad(set_to_none=True)
        loss = criterion(fp32_model(data), target)
        loss.backward()
        fp32_opt.step()

ptq_model = SimpleQATModel()
ptq_model.load_state_dict(fp32_model.state_dict())
ptq_model.eval()
ptq_fused = torch.ao.quantization.fuse_modules(ptq_model, [["fc1", "relu"]])
ptq_fused.qconfig = torch.ao.quantization.get_default_qconfig("fbgemm")
torch.ao.quantization.prepare(ptq_fused, inplace=True)
with torch.no_grad():
    for i, (data, _) in enumerate(train_loader):
        ptq_fused(data)
        if i >= 10:
            break
ptq_int8 = torch.ao.quantization.convert(ptq_fused, inplace=False)

model_fused.eval()
qat_int8 = torch.ao.quantization.convert(model_fused, inplace=False)

results = [
    ("FP32 Baseline", fp32_model),
    ("Static PTQ (int8)", ptq_int8),
    ("True QAT (int8)", qat_int8),
]

print(f"{'Model':<20} | {'Accuracy':>9} | {'Size (KB)':>10} | {'Batch Latency':>14}")
print("-" * 62)
for name, m in results:
    acc = evaluate(m, test_loader)
    sz = model_size_kb(m)
    lat = measure_latency_ms(m)
    print(f"{name:<20} | {acc:>8.2%} | {sz:>8.1f} KB | {lat:>11.2f} ms")

/tmp/ipykernel_791/4018323153.py:44: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare(ptq_fused, inplace=True)
/opt/conda/lib/python3.12/site-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release

Model                |  Accuracy |  Size (KB) |  Batch Latency
--------------------------------------------------------------
FP32 Baseline        |   97.55% |    797.2 KB |        1.33 ms
Static PTQ (int8)    |   97.60% |    208.5 KB |        1.26 ms
True QAT (int8)      |   97.55% |    208.5 KB |        0.75 ms


## Findings & Takeaways

1. **FakeQuantize Simulation:** Calibrating a `MinMaxObserver` (`quant_min=0, quant_max=255`) mapped `[0.4, 1.2, -0.9, 0.0]` to `[0.3922, 1.1922, -0.8941, 0.0000]` with a maximum rounding error of `0.0078`, emulating `int8` quantization noise in FP32 during forward passes while keeping gradients in FP32.
2. **QAT Training Convergence:** Fusing `fc1 + relu` into `LinearReLU` and training with `fbgemm` QAT observers drove cross-entropy loss down smoothly across 3 epochs (`0.3175` $\rightarrow$ `0.1168` $\rightarrow$ `0.0773`).
3. **FP32 vs. Static PTQ vs. True QAT Benchmark (MNIST Test Set):**

| Model | Test Accuracy | Checkpoint Size | Batch Latency (256 imgs) |
| :--- | :---: | :---: | :---: |
| **FP32 Baseline** | `97.55%` | `797.2 KB` | `1.33 ms` |
| **Static PTQ (`int8`)** | `97.60%` | `208.5 KB` | `1.26 ms` |
| **True QAT (`int8`)** | `97.55%` | `208.5 KB` | `0.75 ms` |

4. **Compression & Speedup:** Converting the QAT-prepared network to true `int8` reduced storage footprint by **3.8×** (`797.2 KB` down to `208.5 KB`) and accelerated CPU batch inference by **1.77×** (`1.33 ms` down to `0.75 ms`) with **0.00% accuracy degradation** relative to the FP32 baseline.